# AI Lead Desk — LLM and Admissions Counselor

## Objective
Build the actual AI admissions counselor using LangChain
and an API-based Large Language Model (LLM).

## Responsibilities
- Connect to the hosted LLM securely.
- Conduct natural admissions conversations.
- Follow institute-approved counselling rules.
- Use conversation history to maintain context.
- Prepare for integration with the institute knowledge base (RAG).
- Support English, Hindi, Marathi and mixed-language conversations.

## Current stage
LLM connection and conversational behaviour.

## Later stages
Knowledge retrieval, student profiling, course recommendations,
objection detection and integration with the lead management system.

In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# Load API credentials from the project's .env file, if present.
load_dotenv()

# Check whether the API key is available without displaying it.
if os.getenv("GROQ_API_KEY"):
    print("Groq API key found.")
else:
    print("Groq API key not found. Configure GROQ_API_KEY in your .env file.")

Groq API key found.


In [2]:
# Initialize the Groq-hosted LLM through LangChain.

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2,
    max_tokens=512,
    max_retries=2
)

# Test the actual model connection.
response = llm.invoke(
    "Introduce yourself as an AI admissions counselor in one sentence."
)

print("Model initialized:", llm.model_name)
print("\nAI response:")
print(response.content)

Model initialized: openai/gpt-oss-120b

AI response:
I’m an AI admissions counselor here to guide you through every step of the application process with personalized, data‑driven advice.


In [3]:
from langchain_core.messages import SystemMessage, HumanMessage

COUNSELOR_SYSTEM_PROMPT = """
You are the AI admissions counselor for an educational institute.

Your responsibilities:
1. Have natural, helpful conversations with prospective students.
2. Understand their background, education, skills, goals and constraints.
3. Support English, Hindi, Marathi and mixed-language conversations.
4. Ask relevant follow-up questions instead of interrogating the student.
5. Explain suitable next steps clearly and politely.

Accuracy rules:
- Never invent institute fees, discounts, schedules or placement guarantees.
- Use verified institute information only when answering institute-specific questions.
- If the required information is missing, explain that it cannot be verified
  and offer to connect the student to a human counselor.
- Never guarantee admission, employment or career outcomes.
- Keep responses concise and conversational.

Available verified institute information:
{institute_context}
"""


def counsel_once(student_message: str, institute_context: str = ""):
    """Generate a genuine counselor response using the configured LLM."""

    context = institute_context.strip()

    if not context:
        context = "No verified institute information has been provided yet."

    system_prompt = COUNSELOR_SYSTEM_PROMPT.format(
        institute_context=context
    )

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=student_message)
    ]

    return llm.invoke(messages)


# Test an actual counselling conversation
student_message = "मला IT मध्ये career करायचं आहे. मी कुठून सुरुवात करू?"

response = counsel_once(student_message)

print("Student:", student_message)
print("\nAI Counselor:", response.content)

Student: मला IT मध्ये career करायचं आहे. मी कुठून सुरुवात करू?

AI Counselor: नमस्कार! IT‑मध्ये करिअर सुरू करायचा विचार छान आहे. थोडं अधिक जाणून घेऊ शकलो तर मी तुम्हाला अधिक योग्य मार्गदर्शन देऊ शकेन.

1. **सध्याची शैक्षणिक स्थिती** – तुम्ही सध्या कोणत्या वर्ग/कोर्समध्ये आहात? (उदा. 12वी, पदवी, डिप्लोमा इ.)

2. **आवड आणि लक्ष्य** – IT मध्ये तुम्हाला कोणत्या क्षेत्रात काम करायचे आहे?  
   - सॉफ्टवेअर डेव्हलपमेंट (वेब/मोबाइल)  
   - डेटा सायन्स / मशीन लर्निंग  
   - नेटवर्किंग / सायबर‑सुरक्षा  
   - क्लाउड / DevOps इ.

3. **पूर्वीचा अनुभव** – काही प्रोग्रॅमिंग, इंटर्नशिप किंवा प्रोजेक्ट्स केले आहेत का?

4. **समय व आर्थिक मर्यादा** – तुम्ही पूर्णवेळ, अर्धवेळ, किंवा ऑनलाइन कोर्सेस घेऊ शकता? बजेटबद्दल काही मर्यादा आहेत का?

5. **भाषा व स्थान** – तुम्हाला इंग्रजी, हिंदी किंवा मराठीमध्ये शिकणे सोयीचे आहे? तुम्ही कोणत्या शहरात किंवा प्रदेशात राहता (ऑनलाइन/ऑफलाइन पर्यायांसाठी)?

हे काही मूलभूत प्रश्न आहेत—तुम्ही उत्तर दिल्यावर मी तुम्हाला योग्य कोर्स, प्रमाणपत्रे, आणि पुढील पायऱ्यांची शिफारस कर

## Conversation Memory

In [4]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Store conversation histories separately for each student/session.
conversation_history = {}


def counsel_chat(
    conversation_id: str,
    student_message: str,
    institute_context: str = ""
):
    """Generate an AI counselor reply while preserving conversation history."""

    system_prompt = COUNSELOR_SYSTEM_PROMPT.format(
        institute_context=(
            institute_context.strip()
            or "No verified institute information has been provided yet."
        )
    )

    # Initialize memory for a new conversation.
    if conversation_id not in conversation_history:
        conversation_history[conversation_id] = [
            SystemMessage(content=system_prompt)
        ]
    else:
        # Refresh the system instructions/context for this turn.
        conversation_history[conversation_id][0] = SystemMessage(
            content=system_prompt
        )

    messages = conversation_history[conversation_id]

    # Add the student's latest message.
    messages.append(HumanMessage(content=student_message))

    try:
        response = llm.invoke(messages)
    except Exception:
        # Remove the unanswered user message if the model request fails.
        messages.pop()
        raise

    # Save the AI response so future turns have context.
    messages.append(AIMessage(content=response.content))

    return response.content


# Test a second turn in the same conversation.
conversation_id = "rahul_demo"

reply = counsel_chat(
    conversation_id,
    "मी सध्या कॉलेजच्या पहिल्या वर्षात आहे आणि मला coding चा अनुभव नाही."
)

print("Student:", "मी सध्या कॉलेजच्या पहिल्या वर्षात आहे आणि मला coding चा अनुभव नाही.")
print("\nAI Counselor:", reply)
print("\nMessages in conversation:", len(conversation_history[conversation_id]) - 1)

Student: मी सध्या कॉलेजच्या पहिल्या वर्षात आहे आणि मला coding चा अनुभव नाही.

AI Counselor: नमस्कार! कॉलेजच्या पहिल्या वर्षात असताना कोडिंगचा अनुभव नसणे पूर्णपणे सामान्य आहे—आता सुरू करायचे असलेली उत्तम वेळ आहे. 😊

थोडं अधिक समजून घेण्यासाठी काही प्रश्न आहेत:

1. **कोणत्या प्रकारचा कोडिंग किंवा टेक्नॉलॉजी क्षेत्रात तुम्हाला रस आहे?**  
   (उदा. वेब डेव्हलपमेंट, मोबाइल अॅप्स, डेटा सायन्स, गेम डिझाइन, इ.)

2. **तुम्ही किती वेळ (सप्ताहात) अभ्यासासाठी देऊ शकता?**  
   (फुल‑टाईम, पार्ट‑टाईम, किंवा फक्त काही तास/आठवड्यात)

3. **तुमचे भविष्यातील करिअर लक्ष्य काय आहे?**  
   (उदा. सॉफ्टवेअर इंजिनिअर बनायचे, स्टार्ट‑अप सुरू करायचा, इ.)

4. **कोणते विशेष मर्यादा किंवा प्राधान्य आहेत?**  
   (जसे की बजेट, स्थान, ऑनलाइन/ऑफलाइन शिकणे इ.)

हे उत्तर दिल्यास मी तुम्हाला योग्य कोर्स, लर्निंग पाथ आणि पुढील पायऱ्यांबद्दल अधिक स्पष्ट सूचना देऊ शकेन. तुम्हाला काही विशिष्ट प्रश्न असतील तर नक्की विचारा!

Messages in conversation: 2


In [5]:
from langchain_groq import ChatGroq

# Reinitialize the same model with a larger response budget.
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2,
    max_tokens=1024,
    max_retries=2
)

print("Model:", llm.model_name)
print("Maximum response tokens: 1024")
print("Conversation function will use this updated model.")

Model: openai/gpt-oss-120b
Maximum response tokens: 1024
Conversation function will use this updated model.


In [6]:
# Continue the same conversation created in Cell 5.
student_message = (
    "मला coding सुरू करायचं आहे. माझ्यासाठी सुरुवात कशी करू?"
)

reply = counsel_chat(
    conversation_id="rahul_demo",
    student_message=student_message
)

print("Student:", student_message)
print("\nAI Counselor:", reply)
print("\nConversation messages:",
      len(conversation_history["rahul_demo"]) - 1)

Student: मला coding सुरू करायचं आहे. माझ्यासाठी सुरुवात कशी करू?

AI Counselor: नमस्कार! कोडिंग सुरू करायची इच्छा खूपच छान आहे. खाली काही सोप्या पायऱ्या दिल्या आहेत ज्या तुम्हाला व्यवस्थित पायाभरणी करायला मदत करतील:

| पायरी | काय करायचं? | शिफारस केलेले साधन/स्रोत |
|------|------------|--------------------------|
| 1️⃣ **उद्दिष्ट ठरवा** | तुम्हाला कोणत्या क्षेत्रात (वेब, मोबाइल, डेटा, गेम इ.) काम करायचं आहे? | थोडं विचार करा – नंतर भाषा आणि कोर्स निवडणे सोपं होईल. |
| 2️⃣ **भाषा निवडा** | नवशिक्यांसाठी सर्वात सोपी आणि व्यापक वापरली जाणारी भाषा **Python** आहे. <br>वेबसाठी **HTML/CSS + JavaScript**, मोबाइलसाठी **Kotlin/Swift** इ. | • Python – “Automate the Boring Stuff” (पुस्तक) <br>• JavaScript – freeCodeCamp.org <br>• Kotlin – Kotlin Academy (YouTube) |
| 3️⃣ **मूलभूत संकल्पना शिका** | व्हेरिएबल्स, लूप्स, कंडिशनल्स, फंक्शन्स, डेटा स्ट्रक्चर्स (लिस्ट, डिक्शनरी) इ. | • Codecademy (फ्री बेसिक कोर्स) <br>• Coursera – “Programming for Everybody (Python)” <br>• YouTube चॅनेल “CS50” (इंग्रज

In [7]:
import json
from pathlib import Path

# Find the shared data file from the current folder or its parent.
search_locations = [
    Path.cwd(),
    *Path.cwd().parents
]

DATA_PATH = next(
    (
        folder / "data" / "institute_data.json"
        for folder in search_locations
        if (folder / "data" / "institute_data.json").is_file()
    ),
    None
)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find data/institute_data.json. "
        "Check that this notebook is inside the AI Lead Desk project."
    )

# Load the institute's records and rules.
with DATA_PATH.open("r", encoding="utf-8") as file:
    shared_data = json.load(file)

INSTITUTE_DATA = shared_data["institute"]
COUNSELOR_RULES = shared_data["counselor_rules"]
COURSES = INSTITUTE_DATA["courses"]

print("Institute data loaded successfully.")
print("Institute:", INSTITUTE_DATA["name"])
print("Courses available:", len(COURSES))
print("Source:", DATA_PATH)
print("Demo data:", INSTITUTE_DATA.get("is_sample_data", False))

Institute data loaded successfully.
Institute: Demo Learning Institute
Courses available: 3
Source: c:\Hehe\AI-Lead-Desk-Runtime-Errors\data\institute_data.json
Demo data: True


In [8]:
import re
import json

STOPWORDS = {
    "what", "is", "the", "for", "of", "in", "on", "to", "and",
    "a", "an", "me", "my", "your", "you", "tell", "about",
    "please", "can", "could", "course", "courses", "details",
    "fee", "fees", "price", "duration", "schedule", "timing",
    "chi", "cha", "ki", "ka", "hai", "hai?"
}


def retrieve_institute_context(query: str, top_k: int = 3) -> str:
    """Retrieve matching course records from the shared institute data."""

    query_tokens = {
        token for token in re.findall(r"[a-zA-Z0-9]+", query.casefold())
        if len(token) > 1 and token not in STOPWORDS
    }

    if not query_tokens:
        return ""

    scored_courses = []

    for course in COURSES:
        name_tokens = set(
            re.findall(r"[a-zA-Z0-9]+", course["name"].casefold())
        )

        searchable_text = " ".join([
            course["description"],
            course["duration"],
            course["schedule"],
            course["eligibility"],
            " ".join(course["syllabus"])
        ]).casefold()

        other_tokens = set(re.findall(r"[a-zA-Z0-9]+", searchable_text))

        # Course-name matches receive more weight than general text matches.
        score = (
            3 * len(query_tokens & name_tokens)
            + len(query_tokens & other_tokens)
        )

        if score > 0:
            scored_courses.append((score, course))

    scored_courses.sort(key=lambda item: item[0], reverse=True)

    records = [
        course for _, course in scored_courses[:top_k]
    ]

    if not records:
        return ""

    return json.dumps(
        {
            "institute": INSTITUTE_DATA["name"],
            "courses": records,
            "counselor_rules": COUNSELOR_RULES
        },
        ensure_ascii=False,
        indent=2
    )


# Test retrieval using a question about a specific course.
retrieved_context = retrieve_institute_context(
    "What is the fee and duration of Python Programming?"
)

if retrieved_context:
    print(retrieved_context)
else:
    print("No relevant institute records retrieved.")

{
  "institute": "Demo Learning Institute",
  "courses": [
    {
      "course_id": "PY001",
      "name": "Python Programming",
      "description": "Beginner-friendly Python programming course.",
      "fee_inr": 8000,
      "duration": "3 months",
      "schedule": "Weekends",
      "eligibility": "No prior programming experience required",
      "syllabus": [
        "Python fundamentals",
        "Variables and data types",
        "Conditions and loops",
        "Functions",
        "Lists and dictionaries",
        "File handling"
      ]
    },
    {
      "course_id": "DS001",
      "name": "Data Science Foundations",
      "description": "Introduction to data analysis and machine learning.",
      "fee_inr": 15000,
      "duration": "6 months",
      "schedule": "Monday, Wednesday and Friday",
      "eligibility": "Basic computer skills recommended",
      "syllabus": [
        "Python for data analysis",
        "NumPy",
        "Pandas",
        "Data visualization",
      

In [9]:
# Enforce database-grounded answers.
# The model must not answer factual questions without retrieved records.

STRICT_COUNSELOR_PROMPT = """
You are the admissions counselor for an educational institute.

STRICT SOURCE RULES:
1. Use only facts explicitly supported by the retrieved institute records.
2. Do not use pretrained knowledge to add facts, recommendations,
   fees, discounts, schedules or placement claims.
3. Do not invent missing information or make assumptions.
4. If the records do not answer the question, clearly state that
   the information cannot be verified and offer human assistance.
5. You may use natural, friendly language and match the student's
   language, but all factual claims must be supported by the records.
6. Never claim that a booking or other action was completed unless
   the backend confirms it.

RETRIEVED INSTITUTE RECORDS:
{institute_context}
"""


def database_grounded_counsel(
    conversation_id: str,
    student_message: str
) -> str:
    """
    Retrieve institute records first.
    Do not call the LLM if no relevant records are found.
    """

    context = retrieve_institute_context(student_message)

    if not context:
        return (
            "I couldn't verify this information from the institute's "
            "available records. Please specify the course, or I can "
            "connect you with an admissions counselor."
        )

    # Update the counselor instructions for this grounded interaction.
    global COUNSELOR_SYSTEM_PROMPT
    previous_prompt = COUNSELOR_SYSTEM_PROMPT
    COUNSELOR_SYSTEM_PROMPT = STRICT_COUNSELOR_PROMPT

    try:
        return counsel_chat(
            conversation_id=conversation_id,
            student_message=student_message,
            institute_context=context
        )
    finally:
        # Restore the original prompt for other notebook experiments.
        COUNSELOR_SYSTEM_PROMPT = previous_prompt


# Test with a question supported by the sample records.
test_question = "What is the fee and duration of Python Programming?"

answer = database_grounded_counsel(
    conversation_id="grounded_test_001",
    student_message=test_question
)

print("Student:", test_question)
print("\nDatabase-grounded AI response:")
print(answer)

Student: What is the fee and duration of Python Programming?

Database-grounded AI response:
The **Python Programming** course costs **₹8,000** and runs for **3 months**.


In [10]:
# Test a question that is not supported by the available course records.
test_question = (
    "Does the Robotics course guarantee 100% job placement?"
)

# This function retrieves data first and skips the LLM if nothing matches.
answer = database_grounded_counsel(
    conversation_id="grounded_test_002",
    student_message=test_question
)

print("Student:", test_question)
print("\nCounselor response:")
print(answer)

Student: Does the Robotics course guarantee 100% job placement?

Counselor response:
I couldn't verify this information from the institute's available records. Please specify the course, or I can connect you with an admissions counselor.


In [11]:
# Identify the type of factual information requested by the student.
FACT_KEYWORDS = {
    "fee_inr": [
        "fee", "fees", "price", "cost", "फी", "फीस", "शुल्क"
    ],
    "duration": [
        "duration", "how long", "months", "कालावधी", "किती महिने"
    ],
    "schedule": [
        "schedule", "timing", "timings", "batch", "वेळापत्रक", "वेळ"
    ],
    "eligibility": [
        "eligibility", "eligible", "qualification", "पात्रता"
    ],
    "syllabus": [
        "syllabus", "curriculum", "topics", "अभ्यासक्रम"
    ],
    "placement_policy": [
        "placement", "job guarantee", "guaranteed job",
        "100% job", "नोकरीची हमी", "नोकरी मिळेल"
    ]
}


def detect_requested_facts(query: str) -> list[str]:
    """Identify factual fields explicitly asked about."""
    query = query.casefold()

    return [
        field
        for field, keywords in FACT_KEYWORDS.items()
        if any(keyword.casefold() in query for keyword in keywords)
    ]


def find_course_in_query(query: str) -> dict | None:
    """Find a course using its name or distinctive name words."""
    query = query.casefold()

    # Prefer a complete course name or course ID match.
    for course in COURSES:
        if (
            course["name"].casefold() in query
            or course["course_id"].casefold() in query
        ):
            return course

    # Otherwise match distinctive words from the course name.
    query_words = set(re.findall(r"[a-zA-Z0-9]+", query))
    candidates = []

    for course in COURSES:
        name_words = set(
            re.findall(r"[a-zA-Z0-9]+", course["name"].casefold())
        )
        score = len(query_words & name_words)

        if score:
            candidates.append((score, course))

    candidates.sort(key=lambda item: item[0], reverse=True)

    if not candidates:
        return None

    # Avoid guessing when two courses match equally well.
    if len(candidates) > 1 and candidates[0][0] == candidates[1][0]:
        return None

    return candidates[0][1]


def verify_requested_facts(query: str) -> dict:
    """
    Check whether the requested facts exist in the matched course record.
    This runs before the LLM is called.
    """
    fields = detect_requested_facts(query)
    course = find_course_in_query(query)

    if not fields:
        return {
            "allowed": True,
            "reason": "No specific factual field detected."
        }

    if course is None:
        return {
            "allowed": False,
            "reason": "The course could not be identified reliably."
        }

    missing = [
        field for field in fields
        if course.get(field) is None or course.get(field) == ""
    ]

    if missing:
        return {
            "allowed": False,
            "course": course["name"],
            "missing_fields": missing,
            "reason": "The requested information is missing from the database."
        }

    return {
        "allowed": True,
        "course": course["name"],
        "verified_fields": fields
    }


# Test the case where the course exists but its placement policy is absent.
test_question = "Does Python Programming guarantee 100% job placement?"

check = verify_requested_facts(test_question)

print("Question:", test_question)
print("Verification:", check)

Question: Does Python Programming guarantee 100% job placement?
Verification: {'allowed': False, 'course': 'Python Programming', 'missing_fields': ['placement_policy'], 'reason': 'The requested information is missing from the database.'}


In [24]:

# Store only conversations handled by this strict, grounded counselor.
grounded_conversation_history = {}

MISSING_DATA_REPLY = (
    "I couldn't verify this information from the institute's records. "
    "Would you like me to connect you with an admissions counselor?"
)


def strict_database_counsel(
    conversation_id: str,
    student_message: str
) -> str:
    """
    Answer only when relevant institute records are available
    and any explicitly requested facts are verified.
    """

    # 1. Check that the specific facts requested actually exist.
    verification = verify_requested_facts(student_message)

    if not verification["allowed"]:
        return MISSING_DATA_REPLY

    # 2. Retrieve institute records before calling the LLM.
    context = retrieve_institute_context(student_message)

    if not context:
        return MISSING_DATA_REPLY

    # 3. Build strict instructions using only retrieved records.
    system_prompt = f"""
You are the admissions counselor for {INSTITUTE_DATA['name']}.

STRICT RULES:
- The retrieved institute records are your only source of factual information.
- Do not use pretrained knowledge to add facts or recommendations.
- Do not invent fees, schedules, discounts, policies or placement claims.
- Answer only what the records support.
- If the records do not support an answer, say that you cannot verify it.
- You may paraphrase verified facts naturally.
- Respond in the student's language and conversational register.
- Do not claim that an external action was completed unless confirmed.

RETRIEVED RECORDS:
{context}
"""

    # 4. Load this conversation's previous grounded messages.
    history = grounded_conversation_history.setdefault(
        conversation_id, []
    )

    messages = [
        SystemMessage(content=system_prompt),
        *history[-10:],
        HumanMessage(content=student_message)
    ]

    # 5. Call the actual API model only after the checks pass.
    response = llm.invoke(messages)
    answer = str(response.content).strip()

    # 6. Save the exchange for future conversational context.
    history.extend([
        HumanMessage(content=student_message),
        AIMessage(content=answer)
    ])

    return answer


# Test a supported factual question.
test_question = (
    "Brother What is the fee and duration of Python Programming?"
)

answer = strict_database_counsel(
    conversation_id="strict_test_001",
    student_message=test_question
)

print("Student:", test_question)
print("\nAI Counselor:", answer)

Student: Brother What is the fee and duration of Python Programming?

AI Counselor: Python Programming कोर्स की फीस ₹ 8,000 है और इसका duration 3 months (तीन महीने) है. 🚀


In [25]:
def strict_database_counsel(
    conversation_id: str,
    student_message: str
) -> str:
    """Answer from retrieved institute records and prioritize the current message's language."""

    # 1. Verify that the requested facts exist.
    verification = verify_requested_facts(student_message)

    if not verification["allowed"]:
        return MISSING_DATA_REPLY

    # 2. Retrieve relevant institute records.
    context = retrieve_institute_context(student_message)

    if not context:
        return MISSING_DATA_REPLY

    # 3. Give the latest message priority when choosing the language.
    has_devanagari = any(
        "\u0900" <= char <= "\u097F"
        for char in student_message
    )

    if has_devanagari:
        language_instruction = (
            "Reply in the same language as the latest message. "
            "Determine whether its wording is Marathi or Hindi. "
            "Do not switch languages merely because of earlier messages."
        )
    else:
        language_instruction = (
            "Reply in English if the latest message is in English. "
            "If it is clearly Hindi or Marathi written in Latin letters, "
            "match that language instead. Never infer the reply language "
            "from previous messages alone."
        )

    system_prompt = f"""
You are the admissions counselor for {INSTITUTE_DATA['name']}.

STRICT FACTUAL RULES:
- Use only facts supported by the retrieved institute records below.
- Never invent fees, discounts, schedules or placement claims.
- If the requested information is not supported, say it cannot be verified.
- Do not introduce facts from your pretrained knowledge.
- Do not claim an action was completed without confirmation.

LANGUAGE RULE:
{language_instruction}

RETRIEVED INSTITUTE RECORDS:
{context}
"""

    # 4. Load this conversation's history.
    history = grounded_conversation_history.setdefault(
        conversation_id, []
    )

    messages = [
        SystemMessage(content=system_prompt),
        *history[-10:],
        HumanMessage(content=student_message)
    ]

    # 5. Generate a real response using the API model.
    response = llm.invoke(messages)
    answer = str(response.content).strip()

    # 6. Store the conversation for future turns.
    history.extend([
        HumanMessage(content=student_message),
        AIMessage(content=answer)
    ])

    return answer


# Test in a fresh conversation so previous language preferences do not interfere.
test_question = "Brother, what is the fee and duration of Python Programming?"

answer = strict_database_counsel(
    conversation_id="language_test_english_001",
    student_message=test_question
)

print("Student:", test_question)
print("\nAI Counselor:", answer)

Student: Brother, what is the fee and duration of Python Programming?

AI Counselor: The Python Programming course costs **₹8,000** and runs for **3 months**.


In [26]:
language_tests = [
    (
        "English",
        "What is the fee and duration of Python Programming?"
    ),
    (
        "Hindi",
        "Python Programming की फीस और duration क्या है?"
    ),
    (
        "Marathi",
        "Python Programming ची फी आणि कालावधी किती आहे?"
    )
]

for language, question in language_tests:
    print(f"\n{'=' * 55}")
    print(f"EXPECTED LANGUAGE: {language}")
    print("Student:", question)

    try:
        answer = strict_database_counsel(
            conversation_id=f"language_test_{language.lower()}",
            student_message=question
        )
        print("Counselor:", answer)

    except Exception as error:
        print("Error:", error)


EXPECTED LANGUAGE: English
Student: What is the fee and duration of Python Programming?
Counselor: The Python Programming course costs **₹8,000** and has a **duration of 3 months**.

EXPECTED LANGUAGE: Hindi
Student: Python Programming की फीस और duration क्या है?
Counselor: Python Programming कोर्स की फीस ₹ 8,000 है और इसकी अवधि **3 महीने** की है।

EXPECTED LANGUAGE: Marathi
Student: Python Programming ची फी आणि कालावधी किती आहे?
Counselor: Python Programming कोर्सची फी **₹8,000** आहे आणि त्याचा कालावधी **३ महिने** आहे.


In [28]:
# Expand factual-field detection, including employment and discounts.
FACT_KEYWORDS = {
    "fee_inr": [
        "fee", "fees", "price", "cost", "tuition",
        "फीस", "फी", "शुल्क"
    ],
    "duration": [
        "duration", "how long", "months", "कालावधी", "किती महिने"
    ],
    "schedule": [
        "schedule", "timing", "timings", "batch",
        "वेळापत्रक", "वेळ"
    ],
    "eligibility": [
        "eligibility", "eligible", "qualification", "पात्रता"
    ],
    "syllabus": [
        "syllabus", "curriculum", "topics", "अभ्यासक्रम"
    ],
    "placement_policy": [
        "placement", "job", "employment", "career guarantee",
        "job guarantee", "guarantee a job", "guaranteed job",
        "100% job", "नोकरी", "नोकरीची हमी", "प्लेसमेंट"
    ],
    "discount_policy": [
        "discount", "discounts", "offer", "offers",
        "special price", "scholarship", "सवलत", "छूट"
    ]
}


def detect_requested_facts(query: str) -> list[str]:
    """Identify factual fields requested in a student message."""
    query = query.casefold()

    return [
        field
        for field, keywords in FACT_KEYWORDS.items()
        if any(keyword.casefold() in query for keyword in keywords)
    ]


def verify_requested_facts(query: str) -> dict:
    """Block questions if the requested course or fact is unavailable."""
    fields = detect_requested_facts(query)
    course = find_course_in_query(query)

    if not fields:
        return {
            "allowed": True,
            "reason": "No recognized factual field detected."
        }

    if course is None:
        return {
            "allowed": False,
            "reason": "The course could not be identified reliably."
        }

    missing_fields = [
        field for field in fields
        if course.get(field) is None or course.get(field) == ""
    ]

    if missing_fields:
        return {
            "allowed": False,
            "course": course["name"],
            "missing_fields": missing_fields,
            "reason": "Requested information is missing from institute data."
        }

    return {
        "allowed": True,
        "course": course["name"],
        "verified_fields": fields
    }


# Test the three failure cases without calling the LLM.
safety_tests = [
    "What is the fee for the Robotics course?",
    "Does Python Programming guarantee a job?",
    "What discount is available for Python Programming?"
]

for question in safety_tests:
    result = verify_requested_facts(question)
    print("\nQuestion:", question)
    print("Verification:", result)

    assert not result["allowed"], (
        f"Unsupported question was incorrectly allowed: {question}"
    )

print("\nAll three missing-information checks passed.")


Question: What is the fee for the Robotics course?
Verification: {'allowed': False, 'reason': 'The course could not be identified reliably.'}

Question: Does Python Programming guarantee a job?
Verification: {'allowed': False, 'course': 'Python Programming', 'missing_fields': ['placement_policy'], 'reason': 'Requested information is missing from institute data.'}

Question: What discount is available for Python Programming?
Verification: {'allowed': False, 'course': 'Python Programming', 'missing_fields': ['discount_policy'], 'reason': 'Requested information is missing from institute data.'}

All three missing-information checks passed.


In [29]:
safety_tests = [
    (
        "Missing course",
        "What is the fee for the Robotics course?"
    ),
    (
        "Missing placement policy",
        "Does Python Programming guarantee a job?"
    ),
    (
        "Missing discount policy",
        "What discount is available for Python Programming?"
    )
]

for test_name, question in safety_tests:
    verification = verify_requested_facts(question)

    # This function should return before calling the LLM
    # whenever the required records are missing.
    answer = strict_database_counsel(
        conversation_id=f"safety_{test_name.replace(' ', '_')}",
        student_message=question
    )

    print(f"\n--- {test_name} ---")
    print("Question:", question)
    print("Database check:", verification["reason"])
    print("Counselor:", answer)

    assert not verification["allowed"], (
        f"Safety check unexpectedly allowed: {question}"
    )

    assert answer == MISSING_DATA_REPLY, (
        f"Expected the controlled fallback: {question}"
    )

print("\nAll database-only safety checks passed.")


--- Missing course ---
Question: What is the fee for the Robotics course?
Database check: The course could not be identified reliably.
Counselor: I couldn't verify this information from the institute's records. Would you like me to connect you with an admissions counselor?

--- Missing placement policy ---
Question: Does Python Programming guarantee a job?
Database check: Requested information is missing from institute data.
Counselor: I couldn't verify this information from the institute's records. Would you like me to connect you with an admissions counselor?

--- Missing discount policy ---
Question: What discount is available for Python Programming?
Database check: Requested information is missing from institute data.
Counselor: I couldn't verify this information from the institute's records. Would you like me to connect you with an admissions counselor?

All database-only safety checks passed.
